# weight norm regularization — 验证 lab

对应叶子: `retrieval-recommendation/ranking/weight-norm-regularization`

本 notebook 用于**用代码检验理解**，不是复现教程。至少包含一个失败模式实验或一个数量级估算。

In [ ]:
# 环境自检
import sys
print(sys.version)
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'mps', torch.backends.mps.is_available())
except ImportError:
    print('torch 未安装')

## 假设 (Hypothesis)

数据基本来自 `y=2x`，但训练集最后一条是离群点。预测：未正则化的线性模型会用较大的权重追随离群点；L2 项会缩小权重，并可能降低干净验证集的误差。lambda 太大时会欠拟合。

In [ ]:
import torch

torch.manual_seed(0)
# 11 条干净样本 + 1 条刻意制造的离群样本。shape: [12, 1]
x_train = torch.linspace(-1, 1, 12).unsqueeze(1)
y_train = 2 * x_train
y_train[-1] = 8.0
x_valid = torch.linspace(-1, 1, 101).unsqueeze(1)
y_valid = 2 * x_valid

def fit(lam, steps=300, lr=0.08):
    # w 是一维参数；这里把 L2 regularization 显式写入 loss。
    w = torch.zeros(1, requires_grad=True)
    for _ in range(steps):
        pred = x_train * w
        data_loss = ((pred - y_train) ** 2).mean()
        objective = data_loss + 0.5 * lam * (w ** 2).sum()
        objective.backward()
        with torch.no_grad():
            w -= lr * w.grad
            w.grad.zero_()
    valid_mse = ((x_valid * w - y_valid) ** 2).mean()
    return float(w.detach()), float(data_loss.detach()), float(valid_mse.detach())

for lam in (0.0, 0.1, 1.0, 10.0):
    w, train_mse, valid_mse = fit(lam)
    print(f'lambda={lam:>4}: w={w:6.3f}, ||w||2={abs(w):6.3f}, ' 
          f'train_mse={train_mse:7.4f}, clean_valid_mse={valid_mse:7.4f}')


## 观察与解释

- 比较 `w` 与 `||w||2`：L2 项确实改变了最优参数的大小。
- 比较训练和干净验证 MSE：中等 lambda 在这个特制的含噪 toy 数据上可能更好，过大 lambda 会欠拟合。
- **不能证明**：这不代表某个 lambda 会改善真实推荐系统；真实系统还受 ID 稀疏、负采样、特征尺度、分布漂移和线上反馈影响。
